# Prototype → Production in 30 minutes

A raw LLM call works, but shipping it to production means closing five gates.
This notebook closes them one by one with genai-prod-kit.

1. Observability (cost / tokens / latency)
2. Prompt versioning
3. Regression gate (eval)
4. PII protection

In [9]:
import os, sys
sys.path.insert(0, "../src")
api_key = os.environ["GEMINI_API_KEY"]

## 1. Raw API call — what's wrong with it

First, call the model with no instrumentation. It works, but nothing about
the call is recorded. In production you can't answer "what does this feature
cost per month?" or "why is it slow?"


In [6]:
from genai_prod_kit.providers.gemini import GeminiProvider

provider = GeminiProvider(api_key)
r = provider.generate("What is the capital of Japan?", model="gemini-2.5-flash")
print(r.text)


The capital of Japan is **Tokyo**.


## 2. Route through the Gateway — cost becomes visible

The same call, now through the Gateway. Tokens / cost / latency are written
as one record per call to the sink.

In [ ]:
from genai_prod_kit.gateway import call_llm
from genai_prod_kit.sinks.jsonl import JsonlSink

sink = JsonlSink("nb_invocations.jsonl")
res = call_llm(
    "What is the capital of Japan?",
    provider=provider,
    sink=sink,
    feature="demo",
    model="gemini-2.5-flash",
)
print(res.text)

The capital of Japan is **Tokyo**.


In [8]:
import json

last = json.loads(open("nb_invocations.jsonl", encoding="utf-8").readlines()[-1])
print("tokens :", last["input_tokens"], "/", last["output_tokens"])
print("cost   :", last["estimated_cost_usd"], "USD")
print("latency:", round(last["latency_ms"]), "ms")

tokens : 9 / 8
cost   : 2.2700000000000003e-05 USD
latency: 1617 ms


## 3. Prompt versioning

Prompts live as `{feature}/v{N}.txt`. Recording the version on every call lets
you attribute an accuracy change to a specific prompt version.

In [10]:
from genai_prod_kit.prompts.registry import get_prompt

version, template = get_prompt("toy_summary")
print("active version:", version)
print(template)

active version: v1
次の文章を1行で要約してください: {text}


## 4. Eval gate

Measure accuracy against a golden set. In CI, this gate blocks a merge when
accuracy regresses beyond a threshold.

In [11]:
from genai_prod_kit.evals.runner import load_golden, run_eval

golden = load_golden("../src/genai_prod_kit/evals/golden/toy_sentiment.jsonl")
_, tmpl = get_prompt("toy_sentiment")

def classify(text: str) -> str:
    res = call_llm(
        tmpl.format(text=text),
        provider=provider,
        sink=sink,
        feature="toy_sentiment",
        model="gemini-2.5-flash",
    )
    return res.text.strip().lower()

run = run_eval(golden, classify, feature="toy_sentiment", note="nb")
print(f"accuracy = {run.accuracy:.1%}  ({run.golden_count} items)")

accuracy = 100.0%  (10 items)


## 5. PII protection (shadow → enforce)

Detect and mask PII before sending text to the model. `shadow` only detects
(observe-only); `enforce` rewrites the outgoing text and restores it in the
response.

In [13]:
from genai_prod_kit.pii.pipeline import run_pre_llm

# Tier-1 covers structured PII: email and credit card (Luhn-checked).
sample = "Reach me at taro@example.com or pay with card 4111 1111 1111 1111"

shadow = run_pre_llm(sample, mode="shadow")
print("matches      :", shadow.match_count)
print("sent (shadow):", shadow.redacted_text)    # shadow sends the original

enforce = run_pre_llm(sample, mode="enforce")
print("sent (enforce):", enforce.redacted_text)  # enforce sends the masked text

matches      : 2
sent (shadow): Reach me at taro@example.com or pay with card 4111 1111 1111 1111
sent (enforce): Reach me at <EMAIL_1> or pay with card <CARD_1>


## Wrap-up

We layered observability, versioning, an eval gate, and PII protection onto a
raw call. That's the difference between "it runs" and "it's safe to operate in
production." Notebooks 02–06 drill into each gate individually.